In [0]:
import dlt
from expectation import rules
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType
from modules import replace_comma

In [0]:
@dlt.table(
    name="gold_renewable_energy.gold_monthly_measure",
    comment="Calculations of avg value in the month loaded."
)

def gold_monthly_measure():
    # 1. Read from the upstream DLT silver table dataset
    df_source = dlt.read("silver_renewable_energy.silver_sat_measure")

    # 2. Clean and cast string values to double
    # .withColumn("reduction_double", regexp_replace("reduction_value", ",", ".").cast(DoubleType())) \
    df_cleaned = df_source \
        .withColumn("plant_double", replace_comma("plant_value")) \
        .withColumn("reduction_double", replace_comma("reduction_value")) \
        .withColumn("coordinated_double", replace_comma("coordinated_value"))

    # 3. Group by keys and calculate averages with null handling
    df_aggregated = df_cleaned.groupBy(
        "hash_measure_key",
        month("value_date").alias("month_val"),
        year("value_date").alias("year_val")
    ).agg(
        coalesce(avg("plant_double"), lit(0.0)).alias("avg_real"),
        coalesce(avg("reduction_double"), lit(0.0)).alias("avg_reductions"),
        coalesce(avg("coordinated_double"), lit(0.0)).alias("avg_coordinated")
    )

    # 4. Create final date and delta columns
    final_df = df_aggregated \
        .withColumn("date_", concat_ws("/", col("month_val"), col("year_val"))) \
        .withColumn("diff_real_coordinated", coalesce(col("avg_real") - col("avg_coordinated"), lit(0.0)))

    # 5. Return final schema projection
    return final_df.select(
        "hash_measure_key",
        "date_",
        "avg_real",
        "avg_reductions",
        "avg_coordinated",
        "diff_real_coordinated"
    )

In [0]:
@dlt.table(
    name="gold_renewable_energy.gold_daily_measure",
    comment="Calculations of daily average values."
)
def gold_daily_measure():
    # 1. Read from the upstream DLT silver table dataset
    df_source = dlt.read("silver_renewable_energy.silver_sat_measure")

    # 2. Extract date and convert string values to double
    df_cleaned = df_source \
        .withColumn("date_", to_date(col("value_date"))) \
        .withColumn("plant_double", replace_comma(col("plant_value"))) \
        .withColumn("reduction_double", replace_comma(col("reduction_value"))) \
        .withColumn("coordinated_double", replace_comma(col("coordinated_value")))

    # 3. Group by keys and calculate averages with null handling
    df_aggregated = df_cleaned.groupBy(
        "hash_measure_key",
        "date_"
    ).agg(
        coalesce(avg("plant_double"), lit(0.0)).alias("avg_real"),
        coalesce(avg("reduction_double"), lit(0.0)).alias("avg_reductions"),
        coalesce(avg("coordinated_double"), lit(0.0)).alias("avg_coordinated")
    )

    # 4. Calculate final delta column
    final_df = df_aggregated \
        .withColumn("diff_real_coordinated", coalesce(col("avg_real") - col("avg_coordinated"), lit(0.0)))

    # 5. Return final schema projection
    return final_df.select(
        "hash_measure_key",
        "date_",
        "avg_real",
        "avg_reductions",
        "avg_coordinated",
        "diff_real_coordinated"
    )

In [0]:
@dlt.table(
    name="gold_renewable_energy.gold_weekly_measure",
    comment="Calculations of weekly average values using ISO 8601 week-numbering calendar."
)
def gold_weekly_measure():
    # 1. Read from the upstream DLT silver table dataset
    df_source = dlt.read("silver_renewable_energy.silver_sat_measure")

    # 2. Extract ISO time grains using Spark 3.x+ compatible patterns
    # weekofyear() natively extracts the ISO-8601 week number (1-53)
    # 'Y' represents the ISO week-numbering year
    df_cleaned = df_source \
        .withColumn("iso_week", weekofyear(col("value_date")).cast(StringType())) \
        .withColumn("iso_year", year(col("value_date")).cast(StringType())) \
        .withColumn("plant_double", replace_comma(col("plant_value"))) \
        .withColumn("reduction_double", replace_comma(col("reduction_value"))) \
        .withColumn("coordinated_double", replace_comma(col("coordinated_value")))

    # 3. Group by keys and ISO calendar dimensions to ensure correct data segregation
    df_aggregated = df_cleaned.groupBy(
        "hash_measure_key",
        "iso_week",
        "iso_year"
    ).agg(
        bround(coalesce(avg("plant_double"), lit(0.0)),2).alias("avg_real"),
        bround(coalesce(avg("reduction_double"), lit(0.0)),2).alias("avg_reductions"),
        bround(coalesce(avg("coordinated_double"), lit(0.0)),2).alias("avg_coordinated")
    )

    # 4. Construct the bug-free week/year string and final delta column
    final_df = df_aggregated \
        .withColumn("date_", concat_ws("/", col("iso_week"), col("iso_year"))) \
        .withColumn("diff_real_coordinated", coalesce(col("avg_real") - col("avg_coordinated"), lit(0.0)))

    # 5. Return final schema projection
    return final_df.select(
        "hash_measure_key",
        "date_",
        "avg_real",
        "avg_reductions",
        "avg_coordinated",
        "diff_real_coordinated"
    )

In [0]:
@dlt.table(
    name="gold_renewable_energy.different_real_coordinated",
    comment="Applies rounding and percentage metrics from the daily measure table."
)
def different_real_coordinated():
    # 1. Read from the upstream DLT gold daily table dataset
    df_source = dlt.read("gold_renewable_energy.gold_daily_measure")

    denominator = coalesce(col("avg_real"), lit(0.0))
    # Replicates: ifnull(diff_real_coordinated, 0) * 100
    numerator = coalesce(col("diff_real_coordinated"), lit(0.0)) * 100

    # 3. Apply transformation logic and replication of try_divide
    final_df = df_source \
        .withColumn("value_", bround(col("diff_real_coordinated"), 2)) \
        .withColumn(
            "value_percentage",
            bround(
                when(denominator == 0, lit(0.0))  # Replicates try_divide zero check & outer ifnull fallback
                .otherwise(numerator / denominator),
                2
            )
        )

    # 4. Return final schema projection with SQL aliases
    return final_df.select(
        col("hash_measure_key").alias("hash_link_key"),
        "date_",
        "value_",
        "value_percentage"
    )